In [3]:
import numpy as np

In [ ]:

def transition(state, action, layout, circle):

    def roll_security_dice():
        trap_triggered = False
        dice_roll = np.random.choice([0, 1])
        return (dice_roll, trap_triggered)

    def roll_normal_dice():
        trap_triggered = np.random.choice([True, False])
        dice_roll = np.random.choice([0, 1, 2])
        return (dice_roll, trap_triggered) 


    def roll_risky_dice():
        trap_triggered = True
        dice_roll = np.random.choice([0, 1, 2, 3])
        return (dice_roll, trap_triggered)
    

    roll_dice_functions = {0:roll_security_dice, 1:roll_normal_dice, 2:roll_risky_dice} 


    (current_position, current_skip_next_turn) = state
    new_skip_next_turn = False

    if current_position == 14:
        print("you won, idiot")
        return (current_position, new_skip_next_turn)
    
    if current_skip_next_turn:
        return (current_position, new_skip_next_turn)
    


    # roll dices
    roll_function = roll_dice_functions[action]
    (dice_roll, trap_triggered) =  roll_function()

    # find new position
    if dice_roll==0:
        new_position = current_position
    else:
        if (current_position == 2):
                if np.random.choice([True, False]):
                    new_position = current_position + dice_roll
                else:
                    new_position = 9+dice_roll

        elif  current_position in range(10): # but not 2

            new_position = current_position+dice_roll

            if new_position>=10:
                if circle:
                    new_position -= 10
                else:
                    new_position = 14

        elif current_position in range(10, 14):
            new_position = current_position+dice_roll

            if new_position>=14:
                if circle:
                    new_position -=14
                else:
                    new_position = 14

    # Deal with the traps
    if trap_triggered:

        trap = layout[new_position]

        if trap == 4:
            trap = np.random.choice([1, 2, 3])

        if   trap == 1:
            new_position = 0

        elif trap == 2:
            if new_position in range(10, 13):
                new_position -= 7
            new_position = max(0, new_position - 3)

        elif trap == 3:
            new_skip_next_turn=True

    return (new_position, new_skip_next_turn)
    

        

In [62]:

def expected_policy_value(state, action, V, layout, circle, state_index, alpha):

    def rolls_security_dice():
        possible_trap_triggered = [False]
        possible_dice_rolls = [0, 1]
        p = 1/2
        return [(p, i, j) for i in possible_trap_triggered for j in possible_dice_rolls]

    def rolls_normal_dice():
        possible_trap_triggered = [True, False]
        possible_dice_rolls = [0, 1, 2]
        p = 1/6
        return [(p, i, j) for i in possible_trap_triggered for j in possible_dice_rolls]


    def rolls_risky_dice():
        possible_trap_triggered = [True]
        possible_dice_rolls = [0, 1, 2, 3]
        p = 1/4
        return [(p, i, j) for i in possible_trap_triggered for j in possible_dice_rolls]    

    rolls_dice_functions = {0:rolls_security_dice, 1:rolls_normal_dice, 2:rolls_risky_dice} 



    Vn = 0
    (current_position, current_skip_next_turn) = state

    if current_position == 14:
        new_state = (14, False)
        Vn +=  0 + alpha*V[state_index(new_state)]
        return Vn 
    
    if current_skip_next_turn:
        new_state = (current_position, False)
        Vn += 1 + alpha*V[state_index(new_state)]
        return Vn
    
    # roll dices
    roll_function = rolls_dice_functions[action]
    list_rolls =  roll_function()

    list_new_positions_before_traps = []

    for (p, dice_roll, trap_triggered) in list_rolls:
        # find new position
        if dice_roll==0:
            new_position_before_trap = current_position
            list_new_positions_before_traps.append((p, new_position_before_trap, trap_triggered))
        else:
            if (current_position == 2):
                    new_position_before_trap = current_position + dice_roll
                    list_new_positions_before_traps.append((p*1/2, new_position_before_trap, trap_triggered))
                    new_position_before_trap = 9+dice_roll
                    list_new_positions_before_traps.append((p*1/2, new_position_before_trap, trap_triggered))


            elif  current_position in range(10): # but not 2
                new_position_before_trap = current_position+dice_roll
                if new_position_before_trap>=10:
                    if circle:
                        new_position_before_trap -= 10
                    else:
                        new_position_before_trap = 14
                list_new_positions_before_traps.append((p, new_position_before_trap, trap_triggered))

            elif current_position in range(10, 14):
                new_position_before_trap = current_position+dice_roll
                if new_position_before_trap>=14:
                    if circle:
                        new_position_before_trap -=14
                    else:
                        new_position_before_trap = 14
                list_new_positions_before_traps.append((p, new_position_before_trap, trap_triggered))

    list_new_positions_after_traps = []

    for (p, new_position_before_trap, trap_triggered) in list_new_positions_before_traps:
        # Deal with the traps
        trap = layout[new_position_before_trap]

        if  (not trap_triggered) or (trap == 0):
            new_skip_next_turn = False
            list_new_positions_after_traps.append((p, (new_position_before_trap, new_skip_next_turn)))
        else:
            trap_list = []

            if trap == 4:
                trap_list.append(1)
                trap_list.append(2)
                trap_list.append(3)
                p/=3
            else:
                trap_list.append(trap)
            
            for trap in trap_list:
                if   trap == 1:
                    new_position_after_trap = 0
                    new_skip_next_turn = False

                elif trap == 2:
                    new_position_after_trap = new_position_before_trap
                    if new_position_after_trap in range(10, 13):
                        new_position_after_trap -= 7
                    new_position_after_trap = max(0, new_position_after_trap - 3)
                    new_skip_next_turn = False


                elif trap == 3:
                    new_position_after_trap = new_position_before_trap
                    new_skip_next_turn=True

                list_new_positions_after_traps.append((p, (new_position_after_trap, new_skip_next_turn)))

    # print(list_new_positions_after_traps)
    for (p, new_state) in list_new_positions_after_traps:
        Vn +=  p*(1 + alpha*V[state_index(new_state)])

    return Vn





    
def value_iteration(layout, circle, theta, alpha):

    l = len(layout)

    def state_index(s):
        index = l*s[1]+s[0]
        if type(index) != int:
            print(index)
        return index

    possible_states = []
    for i in range(len(layout)):
        if layout[i]>=3:
            possible_states.append((i, False))
            possible_states.append((i, True))
        else:
            possible_states.append((i, False))

    V = {}

    for state in possible_states:
        V[state_index(state)] = 0
    
    delta = 2*theta
    while delta>=theta:
        print(delta)
        delta = 0
        for state in possible_states:
            v = V[state_index(state)]
            minv = np.inf
            for action in range(3):
                nv = expected_policy_value(state, action, V, layout, circle, state_index, alpha)
                # print(nv)
                if nv <= minv:
                    minv = nv
            V[state_index(state)] = minv
            delta = max(abs(v-V[state_index(state)]),delta)
    print(delta)
    return V
        


In [63]:
# circle: a boolean variable (type bool), indicating if the player must land exactly on
# the final, goal, square 15 to win (circle = True) or still wins by overstepping the final
# square (circle = False).
circle = False

# layout: a vector of type numpy.ndarray that represents the layout of the game, containing 15 values
#         representing the 15 squares of the Snakes and Ladders game:
# layout[i] = 0 if it is an ordinary square
#           = 1 if it is a “restart” trap (go back to square 1)
#           = 2 if it is a “penalty” trap (go back 3 steps)
#           = 3 if it is a “prison” trap (skip next turn)
#           = 4 if it is a “mystery” trap (random effect among the three previous)
# Note that the first and final squares cannot be trapped.

layout = np.ones(15)*4
layout[0] = 0
layout[14] = 0

value_iteration(layout, circle, 0.01, 1)

0.02
2.398148148148148
1.5138888888888888
1.4043209876543208
1.2757201646090532
1.203017832647463
1.1481481481481461
1.1073007163542155
1.077224000406444
1.0502243766829231
1.015173838676354
1.0101158924509015
1.006743928300601
1.0044959522004007
1.0029973014669338
1.0019982009779547
1.0013321339853043
1.0008880893235386
1.0005920595490245
1.0003947063660128
1.0002631375773454
1.0001754250515624
1.000116950034375
1.0000779666895845
1.0000519777930563
1.000034651862034
1.0000231012413607
1.000015400827575
1.0000102672183786
1.0000068448122548
1.000004563208165
1.000003042138779
1.0000020280925241
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
0.9980271431583816
0.9921054674719159
0.9868692427945263
0.9818976675860682
0.9770645009295009
0.9722999964173624
0.96757

0.6872572181952137
0.6821650453213977
0.6771106024617097
0.6720936100589654
0.6671137906271838
0.6621708687364674
0.6572645709976541
0.6523946260472826
0.6475607645324999
0.6427627190962539
0.6380002243623153
0.633273016920981
0.628580835313926
0.6239234200204749
0.6193005134425107
0.6147118598906047
0.6101572055701752
0.605636298566651
0.6011488888322276
0.5966947281718831
0.5922735702293664
0.5878851704740384
0.5835292861868027
0.5792056764472306
0.5749141021197772
0.5706543258408487
0.5664261120055301
0.5622292267546811
0.5580634379618914
0.5539285152204911
0.5498242298313585
0.5457503547895328
0.5417066647722493
0.5376929361261205
0.5337089468550289
0.529754476607593
0.5258293066651447
0.5219332199297071
0.5180660009117162
0.5142274357183112
0.5104173120413975
0.5066354191461926
0.5028815478590047
0.4991554905561486
0.4954570411524628
0.4917859950895149
0.48814214932465916
0.4845253023195255
0.4809352540292764
0.47737180589112427
0.47383476081347453
0.47032392316521054
0.4668390987

{0: 238.070484224759,
 1: 235.59030570273373,
 16: 236.59030570273373,
 2: 230.6197441886448,
 17: 231.6197441886448,
 3: 234.22186429477154,
 18: 235.22186429477154,
 4: 229.12286873023461,
 19: 230.12286873023461,
 5: 221.4098239183131,
 20: 222.4098239183131,
 6: 204.1819400907379,
 21: 205.1819400907379,
 7: 172.2743236709824,
 22: 173.2743236709824,
 8: 112.31310607887423,
 23: 113.31310607887423,
 9: 1.0,
 24: 2.0,
 10: 207.13450553850876,
 25: 208.13450553850876,
 11: 174.94585735651478,
 26: 175.94585735651478,
 12: 113.05124744081695,
 27: 114.05124744081695,
 13: 1.0,
 28: 2.0,
 14: 0}

In [6]:

initial_position = 0
skip_next_turn = False
state = (initial_position, skip_next_turn)

In [7]:
state = transition(state, 0, layout, circle)
state

(0, False)

In [9]:
"""
Determines the optimal strategy regarding the choice of dice in the S&L game

Parameters
----------
layout : numpy.ndarray
    Represents the 15 squares of S&L game, each layout[i] represents the type of square:
        0 = ordinary square
        1 = restart trap (go back to square 1)
        2 = penalty trap (go back 3 steps)
        3 = prison trap (skip next turn)
        4 = mystery trap (random trap)
        
circle : bool
    True: player must land exactly on final square to win, otherwise loop back to square 1
    False: player wins as soon as they land on or overstep the final square

Returns
-------
expec : numpy.ndarray
    expected cost for each square (excl. final/goal square)    

dice : numpy.ndarray
    choice of dice for each square (excl. final/goal square)

"""
def markovDecision(layout,circle):
    # Initialize solution arrays
    expec = np.zeros(14) # index 0 = square1; index 13 = square 14
    dice = np.zeros(14) # goal square does not need an optimal action
    
    # Implement Value-iteration algo and return optimal policy
    
    
    return [expec, dice]

